# 00 — Environnement Python et validation GPU

**Objectif :** vérifier les versions, détecter CUDA et exécuter un calcul réel sur le GPU.

**Entrées :** aucune.  
**Sortie :** `reports/environment.json`.  
**Dépendances :** aucune étape précédente.  
**Temps estimé :** moins de deux minutes après installation.  
**Ressources :** quelques centaines de Mo de VRAM pendant le smoke test.

## Configuration

In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
from pathlib import Path

import torch


def find_project_root(start: Path | None = None) -> Path:
    """Return the nearest parent containing pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Racine du projet introuvable (pyproject.toml absent).")


ROOT = find_project_root()
REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Racine : {ROOT}")
print(f"Python : {sys.version.split()[0]}")
print(f"OS     : {platform.platform()}")
print(f"Torch  : {torch.__version__}")

## Détection NVIDIA et CUDA

In [ ]:
try:
    nvidia_smi = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv,noheader",
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    nvidia_smi = "nvidia-smi indisponible"

cuda_available = torch.cuda.is_available()
device = torch.device("cuda" if cuda_available else "cpu")

environment = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "torch_cuda_runtime": torch.version.cuda,
    "cuda_available": cuda_available,
    "device": str(device),
    "nvidia_smi": nvidia_smi,
}

if cuda_available:
    properties = torch.cuda.get_device_properties(0)
    environment.update(
        {
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_vram_gb": round(properties.total_memory / 1024**3, 2),
            "gpu_compute_capability": f"{properties.major}.{properties.minor}",
        }
    )

print(json.dumps(environment, indent=2, ensure_ascii=False))

## Smoke test de calcul

In [ ]:
torch.manual_seed(42)
if cuda_available:
    torch.cuda.manual_seed_all(42)

matrix = torch.randn((1024, 1024), device=device)
result = matrix @ matrix
if cuda_available:
    torch.cuda.synchronize()

smoke_value = float(result[0, 0].item())
assert torch.isfinite(result).all(), "Le smoke test a produit des valeurs non finies."
environment["smoke_test_value"] = smoke_value
environment["smoke_test_passed"] = True

print(f"Calcul validé sur {device}: {smoke_value:.6f}")

## Sauvegarde du rapport

In [ ]:
report_path = REPORTS_DIR / "environment.json"
report_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Rapport sauvegardé : {report_path}")

## Conclusion

Le pipeline utilisera CUDA lorsque `cuda_available` vaut `true`. Dans le cas contraire,
les étapes restent exécutables sur CPU avec des batchs plus petits.